In [3]:
## first we will read in necessary python libraries and intialize the GEE module
## you will have to authenticate GEE on your machine to use this code

import ee
import matplotlib.pyplot as plt
from datetime import datetime, timezone, timedelta
import geopandas as gpd
from shapely.geometry import box
import pandas as pd
import geemap
import os
import pprint
import time
import json

## if you need to authenticate GEE use this code (will give you a link to allow you to authenticate it online )
# ee.Authenticate()

## initialize the Earth Engine module
ee.Initialize(project='ee-tlasharr')

## get the time period we care about
start_date_data = '2018-01-01'
end_date_data = '2022-12-31'


In [ ]:

## load in a shapefile or area of interest, this is an assest that is already loaded into GEE
md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/WY_Sublette') ## "normal" temperate environment
region = md_cooridor.geometry().bounds()

## load the precipitation, swe, and temperature data from Daymet
precip_dat = ee.ImageCollection("NASA/ORNL/DAYMET_V4") \
                .filterDate(start_date_data, end_date_data) \
                .select('prcp', 'swe', 'tmax', 'tmin')


## clip the modis data to the bounding box
precip_dat = precip_dat.map(lambda image: image.clip(region))

## make sure the time property is there and correct (i.e., dates for the images)
def check_time_start(image):
    return ee.Algorithms.If(
        image.get('system:time_start'),
        image,
        image.set('system:time_start', ee.Date(image.date()).millis())
    )

precip_dat = precip_dat.map(lambda img: ee.Image(check_time_start(img)))
print(precip_dat)


ee.ImageCollection({
  "functionInvocationValue": {
    "functionName": "Collection.map",
    "arguments": {
      "baseAlgorithm": {
        "functionDefinitionValue": {
          "argumentNames": [
            "_MAPPING_VAR_0_0"
          ],
          "body": {
            "functionInvocationValue": {
              "functionName": "If",
              "arguments": {
                "condition": {
                  "functionInvocationValue": {
                    "functionName": "Element.get",
                    "arguments": {
                      "object": {
                        "argumentReference": "_MAPPING_VAR_0_0"
                      },
                      "property": {
                        "constantValue": "system:time_start"
                      }
                    }
                  }
                },
                "falseCase": {
                  "functionInvocationValue": {
                    "functionName": "Element.set",
                    "arguments":

In [9]:
print("Bands available in the first image (Snow):", precip_dat.first().bandNames().getInfo())
print("Number of Images:", precip_dat.size().getInfo())

Bands available in the first image (Snow): ['prcp', 'swe', 'tmax', 'tmin']
Number of Images: 1824


In [16]:
import ee
ee.Initialize()

def export_annual_multiband_image(collection, start_year, end_year, region):
    for year in range(start_year, end_year + 1):
        start_date = f"{year}-01-01"
        end_date = f"{year}-12-31"
        
        # Filter the collection for the year
        yearly_collection = collection.filterDate(start_date, end_date)

        # Convert to a multiband image
        multiband_image = yearly_collection.toBands()

        # Rename bands to include 'doy' (day of year) as suffix
        def rename_band(band, index):
            # Add 1 because 'index' is 0-based and DOY is 1-based
            doy = ee.Number(index).add(1).format('doy%03d')
            return ee.String(band).replace('.*_(.*)', '$1').cat('_').cat(doy)

        band_names = multiband_image.bandNames()
        new_band_names = band_names.map(rename_band)
        multiband_image = multiband_image.rename(new_band_names)

        # Define the export parameters
        export_params = {
            'description': f'Annual_Data_{year}',
            'image': multiband_image,
            'scale': 1000,  # Adjust the scale based on the resolution of your data and needs
            'region': region,
            'fileFormat': 'GeoTIFF',
            'maxPixels': 1e13
        }

        # Start the export task for the multiband image
        task = ee.batch.Export.image.toDrive(**export_params)
        task.start()
        print(f"Export task started for annual multiband image of {year}")

# Your bounding box or specific region
region = ee.FeatureCollection('projects/ee-tlasharr/assets/WY_Sublette').geometry().bounds()

# Your original precip_dat ImageCollection
start_date_data = '2018-01-01'
end_date_data = '2022-12-31'
precip_dat = ee.ImageCollection("NASA/ORNL/DAYMET_V4") \
                .filterDate(start_date_data, end_date_data) \
                .select('prcp', 'swe', 'tmax', 'tmin') \
                .map(lambda image: image.clip(region))

# Execute the export function
export_annual_multiband_image(precip_dat, 2018, 2022, region)


Export task started for annual multiband image of 2018
Export task started for annual multiband image of 2019
Export task started for annual multiband image of 2020
Export task started for annual multiband image of 2021
Export task started for annual multiband image of 2022
